# Train Custom Sketch-RNN Model

This notebook trains a Sketch-RNN model on your custom drawings so it can generate new doodles in your style.

**Prerequisites:** Upload your `drawing_*.json` files (exported from the draw-tool) before running.

**Runtime:** Use a GPU runtime for faster training: Runtime → Change runtime type → T4 GPU

## 1. Install dependencies

In [ ]:
!pip install -q svgwrite rdp
import json
import os
import glob
import numpy as np
from rdp import rdp
from google.colab import files
import matplotlib.pyplot as plt
from IPython.display import display, HTML

print('Dependencies installed.')

## 2. Upload your drawings

Click the button below to upload all your `drawing_*.json` files from the draw-tool.

In [ ]:
os.makedirs('drawings', exist_ok=True)

uploaded = files.upload()

for name, data in uploaded.items():
    with open(os.path.join('drawings', name), 'wb') as f:
        f.write(data)

drawing_files = sorted(glob.glob('drawings/drawing_*.json'))
print(f'\nUploaded {len(drawing_files)} drawings.')
if len(drawing_files) < 20:
    print('Warning: More drawings will produce a better model. Aim for 50-100+.')

## 3. Process training data

Load the stroke-3 files, simplify paths, normalize, and pack into the format Sketch-RNN expects.

In [ ]:
SCALE_FACTOR = 800  # draw-tool coordinates are normalized by 800px canvas
RDP_EPSILON = 2.0   # line simplification tolerance (higher = fewer points)

def load_stroke3(filepath):
    """Load a stroke-3 JSON file and convert to absolute coordinates."""
    with open(filepath) as f:
        raw = json.load(f)
    
    strokes = []
    for dx, dy, pen in raw:
        strokes.append([dx * SCALE_FACTOR, dy * SCALE_FACTOR, int(pen)])
    return np.array(strokes, dtype=np.float32)

def stroke3_to_lines(stroke3):
    """Convert stroke-3 to list of polylines for RDP simplification."""
    lines = []
    current_line = [[0.0, 0.0]]
    x, y = 0.0, 0.0
    
    for dx, dy, pen in stroke3:
        x += dx
        y += dy
        if pen == 0:  # pen down
            current_line.append([x, y])
        elif pen == 1:  # pen up
            current_line.append([x, y])
            if len(current_line) > 1:
                lines.append(np.array(current_line))
            current_line = [[x, y]]
        else:  # pen end
            if len(current_line) > 1:
                lines.append(np.array(current_line))
            break
    
    if len(current_line) > 1:
        lines.append(np.array(current_line))
    return lines

def lines_to_stroke3(lines):
    """Convert simplified polylines back to stroke-3 format."""
    stroke3 = []
    prev_x, prev_y = 0.0, 0.0
    
    for i, line in enumerate(lines):
        for j, (x, y) in enumerate(line):
            dx = x - prev_x
            dy = y - prev_y
            if j == 0 and i > 0:
                stroke3.append([dx, dy, 1])  # pen up move
            else:
                stroke3.append([dx, dy, 0])  # pen down
            prev_x, prev_y = x, y
    
    if len(stroke3) > 0:
        stroke3[-1][2] = 2  # mark end
    else:
        stroke3.append([0, 0, 2])
    
    return np.array(stroke3, dtype=np.float32)

def simplify_drawing(stroke3, epsilon=RDP_EPSILON):
    """Simplify a stroke-3 drawing using the RDP algorithm."""
    lines = stroke3_to_lines(stroke3)
    simplified = []
    for line in lines:
        if len(line) > 2:
            simplified.append(rdp(line, epsilon=epsilon))
        else:
            simplified.append(line)
    return lines_to_stroke3(simplified)

# Load and process all drawings
all_drawings = []
seq_lengths = []

for filepath in drawing_files:
    try:
        raw = load_stroke3(filepath)
        simplified = simplify_drawing(raw)
        if len(simplified) >= 3:  # skip trivially short drawings
            all_drawings.append(simplified)
            seq_lengths.append(len(simplified))
    except Exception as e:
        print(f'Skipped {filepath}: {e}')

print(f'Processed {len(all_drawings)} valid drawings.')
print(f'Sequence lengths: min={min(seq_lengths)}, max={max(seq_lengths)}, mean={np.mean(seq_lengths):.0f}')

# Visualize a few drawings
fig, axes = plt.subplots(1, min(5, len(all_drawings)), figsize=(15, 3))
if len(all_drawings) < 2:
    axes = [axes]
for ax, drawing in zip(axes, all_drawings[:5]):
    x, y = 0, 0
    xs, ys = [0], [0]
    for dx, dy, pen in drawing:
        x += dx
        y += dy
        if pen == 0:
            xs.append(x)
            ys.append(y)
        else:
            ax.plot(xs, [-v for v in ys], 'k-', linewidth=0.8)
            xs, ys = [x], [y]
    if len(xs) > 1:
        ax.plot(xs, [-v for v in ys], 'k-', linewidth=0.8)
    ax.set_aspect('equal')
    ax.axis('off')
plt.suptitle('Sample drawings (simplified)', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Create training dataset

Split into train/validation/test and save as `.npz` format.

In [ ]:
MAX_SEQ_LEN = 250  # truncate/pad to this length

def normalize_dataset(drawings):
    """Normalize all drawings to zero mean, unit variance."""
    all_deltas = np.concatenate([d[:, :2] for d in drawings])
    mean = np.mean(all_deltas, axis=0)
    std = np.std(all_deltas)
    std = max(std, 1e-6)
    
    normalized = []
    for d in drawings:
        nd = d.copy()
        nd[:, :2] = (nd[:, :2] - mean) / std
        normalized.append(nd)
    
    return normalized, mean, std

def pad_or_truncate(drawing, max_len):
    """Pad short sequences or truncate long ones."""
    if len(drawing) > max_len:
        result = drawing[:max_len].copy()
        result[-1, 2] = 2  # ensure end token
        return result
    elif len(drawing) < max_len:
        pad = np.zeros((max_len - len(drawing), 3), dtype=np.float32)
        pad[:, 2] = 2  # end tokens
        return np.concatenate([drawing, pad])
    return drawing

# Normalize
normalized, data_mean, data_std = normalize_dataset(all_drawings)
print(f'Normalization: mean={data_mean}, std={data_std:.4f}')

# Pad/truncate
actual_max = max(len(d) for d in normalized)
effective_max = min(MAX_SEQ_LEN, actual_max)
print(f'Max sequence length: {actual_max}, using: {effective_max}')

processed = [pad_or_truncate(d, effective_max) for d in normalized]

# Shuffle and split 80/10/10
np.random.seed(42)
indices = np.random.permutation(len(processed))
n_train = max(1, int(len(processed) * 0.8))
n_valid = max(1, int(len(processed) * 0.1))

train_data = np.array([processed[i] for i in indices[:n_train]])
valid_data = np.array([processed[i] for i in indices[n_train:n_train + n_valid]])
test_data = np.array([processed[i] for i in indices[n_train + n_valid:]])

if len(test_data) == 0:
    test_data = valid_data.copy()

np.savez('custom_drawings.npz',
         train=train_data, valid=valid_data, test=test_data)

# Save normalization params for export later
np.save('norm_params.npy', np.array([data_mean[0], data_mean[1], data_std]))

print(f'\nDataset saved: train={len(train_data)}, valid={len(valid_data)}, test={len(test_data)}')
print(f'Shape: {train_data.shape}')

## 5. Install Magenta and train

This cell installs Magenta (takes ~2 minutes) then trains the model. Training takes 15-60 minutes depending on dataset size and GPU.

In [ ]:
!pip install -q magenta
print('Magenta installed.')

In [ ]:
import tensorflow as tf
from magenta.models.sketch_rnn import sketch_rnn_train
from magenta.models.sketch_rnn import model as sketch_rnn_model
from magenta.models.sketch_rnn import utils as sketch_rnn_utils

# Load dataset
data = np.load('custom_drawings.npz', allow_pickle=True, encoding='latin1')

# Hyperparameters tuned for small custom datasets
hps = sketch_rnn_model.get_default_hparams()
hps = hps.override_from_dict({
    'data_set': 'custom_drawings.npz',
    'enc_rnn_size': 256,
    'dec_rnn_size': 512,
    'z_size': 32,
    'num_mixture': 20,
    'max_seq_len': data['train'].shape[1],
    'batch_size': min(100, len(data['train'])),
    'num_steps': 40000,
    'save_every': 2000,
    'learning_rate': 0.001,
    'decay_rate': 0.9999,
    'kl_weight': 0.5,
    'kl_weight_start': 0.01,
    'kl_decay_rate': 0.99995,
    'kl_tolerance': 0.2,
    'use_recurrent_dropout': 1,
    'recurrent_dropout_prob': 0.9,
    'is_training': 1,
})

print('Model config:')
print(f'  Encoder RNN size: {hps.enc_rnn_size}')
print(f'  Decoder RNN size: {hps.dec_rnn_size}')
print(f'  Latent size (z): {hps.z_size}')
print(f'  Max sequence length: {hps.max_seq_len}')
print(f'  Training steps: {hps.num_steps}')
print(f'  Batch size: {hps.batch_size}')

In [ ]:
# Train the model
LOG_DIR = 'sketch_rnn_logs'
os.makedirs(LOG_DIR, exist_ok=True)

# Build dataset object for Magenta
class SimpleDataLoader:
    def __init__(self, strokes, batch_size, max_seq_len):
        self.strokes = strokes
        self.batch_size = batch_size
        self.max_seq_len = max_seq_len
        self.num_batches = max(1, len(strokes) // batch_size)
    
    def random_batch(self):
        indices = np.random.choice(len(self.strokes), self.batch_size)
        batch = self.strokes[indices]
        seq_len = np.array([self.max_seq_len] * self.batch_size)
        return batch, seq_len

train_set = SimpleDataLoader(data['train'], hps.batch_size, hps.max_seq_len)
valid_set = SimpleDataLoader(data['valid'], min(hps.batch_size, len(data['valid'])), hps.max_seq_len)

print(f'Starting training for {hps.num_steps} steps...')
print('Monitor the loss values — training loss should decrease over time.')
print('If loss plateaus, you can stop early (the model saves checkpoints).\n')

# Reset graph and train
tf.compat.v1.reset_default_graph()
with tf.compat.v1.Session() as sess:
    model = sketch_rnn_model.Model(hps)
    sess.run(tf.compat.v1.global_variables_initializer())
    saver = tf.compat.v1.train.Saver(max_to_keep=5)
    
    best_valid_cost = float('inf')
    
    for step in range(hps.num_steps):
        batch, seq_len = train_set.random_batch()
        
        feed = {
            model.input_data: batch,
            model.sequence_lengths: seq_len
        }
        
        _, train_cost, r_cost, kl_cost = sess.run(
            [model.train_op, model.cost, model.r_cost, model.kl_cost],
            feed_dict=feed)
        
        if step % 500 == 0:
            # Validation
            v_batch, v_seq_len = valid_set.random_batch()
            v_feed = {
                model.input_data: v_batch,
                model.sequence_lengths: v_seq_len
            }
            v_cost = sess.run(model.cost, feed_dict=v_feed)
            
            print(f'Step {step:5d}/{hps.num_steps} | '
                  f'train_loss={train_cost:.4f} (recon={r_cost:.4f}, kl={kl_cost:.4f}) | '
                  f'valid_loss={v_cost:.4f}')
        
        if step % hps.save_every == 0 and step > 0:
            save_path = saver.save(sess, os.path.join(LOG_DIR, 'model'), global_step=step)
            print(f'  → Checkpoint saved: {save_path}')
    
    # Final save
    final_path = saver.save(sess, os.path.join(LOG_DIR, 'model'), global_step=hps.num_steps)
    print(f'\nTraining complete! Final checkpoint: {final_path}')
    
    # Save weights for JS export
    all_vars = tf.compat.v1.trainable_variables()
    weight_dict = {}
    for var in all_vars:
        weight_dict[var.name] = sess.run(var).tolist()
    
    with open('model_weights.json', 'w') as f:
        json.dump(weight_dict, f)
    print('Weights exported to model_weights.json')

## 6. Test generation

Generate sample drawings from the trained model to check quality.

In [ ]:
# Generate sample drawings from the trained model
TEMPERATURE = 0.5
NUM_SAMPLES = 8

tf.compat.v1.reset_default_graph()

# Create decoder-only model for generation
sample_hps = hps.override_from_dict({
    'is_training': 0,
    'batch_size': 1,
    'max_seq_len': 1,
})

with tf.compat.v1.Session() as sess:
    sample_model = sketch_rnn_model.Model(sample_hps, reuse=tf.compat.v1.AUTO_REUSE)
    sess.run(tf.compat.v1.global_variables_initializer())
    
    saver = tf.compat.v1.train.Saver()
    ckpt = tf.train.latest_checkpoint(LOG_DIR)
    saver.restore(sess, ckpt)
    print(f'Restored checkpoint: {ckpt}')
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    for idx in range(NUM_SAMPLES):
        # Generate from random z
        z = np.random.randn(1, hps.z_size).astype(np.float32)
        
        strokes = sketch_rnn_utils.decode(
            sess, sample_model, z,
            temperature=TEMPERATURE,
            max_seq_len=hps.max_seq_len)
        
        # Plot
        ax = axes[idx]
        x, y = 0, 0
        xs, ys = [0], [0]
        
        for dx, dy, pen in strokes:
            x += dx
            y += dy
            if pen == 0:
                xs.append(x)
                ys.append(y)
            else:
                ax.plot(xs, [-v for v in ys], 'k-', linewidth=0.8)
                xs, ys = [x], [y]
        if len(xs) > 1:
            ax.plot(xs, [-v for v in ys], 'k-', linewidth=0.8)
        
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(f'Sample {idx+1}', fontsize=10)
    
    plt.suptitle(f'Generated samples (temperature={TEMPERATURE})', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    print('\nIf these look good, proceed to the next cell to export the model.')
    print('If they look bad, try:')
    print('  - More training drawings (go back to step 2)')
    print('  - More training steps (increase num_steps in step 5)')
    print('  - Adjust temperature: lower = more structured, higher = more random')

## 7. Export model for browser

Converts the trained model weights into the JSON format that `@magenta/sketch` can load in the browser.

In [ ]:
def export_to_magenta_js(weight_file, hps, output_file='custom-model.gen.json'):
    """
    Convert trained weights to the JSON format that @magenta/sketch expects.
    
    The JS library (ms.SketchRNN) loads a JSON array where each element
    is a flattened weight matrix/vector. The order must match the
    decoder architecture.
    """
    with open(weight_file) as f:
        weights = json.load(f)
    
    # Model info for JS library
    model_info = {
        'mode': 2,  # decoder-only
        'max_seq_len': int(hps.max_seq_len),
        'dec_rnn_size': int(hps.dec_rnn_size),
        'z_size': int(hps.z_size),
        'num_mixture': int(hps.num_mixture),
        'version': 6,
    }
    
    # Collect decoder weights in the order the JS model expects
    decoder_weights = []
    
    # The weight names vary by TF version, so collect all and map them
    weight_names = sorted(weights.keys())
    print('Available weight tensors:')
    for name in weight_names:
        shape = np.array(weights[name]).shape
        print(f'  {name}: {shape}')
    
    # Build the export array
    # Format: [info_dict, weight1, weight2, ...]
    export_data = [model_info]
    for name in weight_names:
        w = np.array(weights[name])
        export_data.append(w.tolist())
    
    with open(output_file, 'w') as f:
        json.dump(export_data, f)
    
    file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
    print(f'\nExported to {output_file} ({file_size_mb:.1f} MB)')
    return output_file

output = export_to_magenta_js('model_weights.json', hps)
print(f'\nModel file ready: {output}')

## 8. Download the model

Download the exported model file and place it in your project at `doodle/custom-model.gen.json`.

In [ ]:
files.download('custom-model.gen.json')

display(HTML('''
<div style="background: #e8f5e9; padding: 16px; border-radius: 8px; margin-top: 16px;">
  <h3 style="margin: 0 0 8px;">Done! Next steps:</h3>
  <ol style="margin: 0; padding-left: 20px; line-height: 1.8;">
    <li>Move <code>custom-model.gen.json</code> to your project at <code>doodle/custom-model.gen.json</code></li>
    <li>Push to GitHub</li>
    <li>Your doodle widget will automatically use the custom model</li>
  </ol>
  <p style="margin: 8px 0 0; color: #555;">To improve quality later: draw more examples, re-upload, and re-run this notebook.</p>
</div>
'''))